# Full Implementation for CPUs and GPUs

This notebook run all the experiments for CPUs and GPUs

In [1]:
import time
import numpy as np
import psutil

import hyperfpga_cluster as hfc
from hyperfpga_cluster import configure_logging
configure_logging()


N_NODES = 4
N_ENGINES_PER_NODE = 4


print(hfc.available_models())
test_nodes = await hfc.request_nodes('4ge21', count=N_NODES)
print(test_nodes)
print(test_nodes[0]['fpga'])

{'4ge21': 0, '3be11': 0}
[{'hostname': 'hyperfpga-4ge21-0-1', 'ip': '192.168.0.3', 'fpga': {'model': '4ge21', 'state': 'unknown', 'firmware': ''}, 'comblocks': {'devs': []}, 'gpios': {'devs': []}, 'dmas': {'devs': []}, 'locked': True, 'locked_by_me': True, 'daemon': {'api_token': 'hyperfpga-4ge21-0-1|falmohammed|1787056897|e77ff4411512acf1dcff88432a6edffbc8648380dcc230187dbdbfb2bf898dd6'}}, {'hostname': 'hyperfpga-4ge21-1-0', 'ip': '192.168.0.6', 'fpga': {'model': '4ge21', 'state': 'unknown', 'firmware': ''}, 'comblocks': {'devs': []}, 'gpios': {'devs': []}, 'dmas': {'devs': []}, 'locked': True, 'locked_by_me': True, 'daemon': {'api_token': 'hyperfpga-4ge21-1-0|falmohammed|1787056897|5a442a3dc6d8b78db5ac1af4273e05f999b415dc60e3c94e1ecde4ba5c4074b4'}}, {'hostname': 'hyperfpga-4ge21-1-1', 'ip': '192.168.0.7', 'fpga': {'model': '4ge21', 'state': 'unknown', 'firmware': ''}, 'comblocks': {'devs': []}, 'gpios': {'devs': []}, 'dmas': {'devs': []}, 'locked': True, 'locked_by_me': True, 'daemon

In [2]:
ALL_RESULTS = {} # gather all results from all experiments

# CPU Part

## Streaming Functions

In [3]:
def row_worker(A_rows, B, K, N):
    """
    Runs on one core. Computes one or more rows of A times B,
    using a systolic array simulation (K,N).

    Returns:
        result_arr: the computed rows
        peak_kb:    memory used
        elapsed_s:  time spent computing only
    """
    
    import psutil
    import os
    import tracemalloc
    import numpy as np
    import time

    tracemalloc.start()

    def PE(a, b):
        """simulate one processing element"""
        return a * b

    def one_row(A_row):
        P = np.zeros(N)
        a_reg = np.zeros(N)
        TOTAL_CYCLES = K + N - 1

        t0 = time.perf_counter()
        for t in range(TOTAL_CYCLES):
            a_new = np.zeros(N)
            for j in range(N):
                a_val = A_row[t] if 0 <= t < K else 0.0
                k = t - j
                b_val = B[k][j] if 0 <= k < K else 0.0
                a_in = a_val if j == 0 else a_reg[j - 1]
                P[j] += PE(a_in, b_val)
                a_new[j] = a_in
            a_reg = a_new
        row_elapsed_s = time.perf_counter() - t0

        return P, row_elapsed_s

    total_elapsed_s = 0
    results = []
    if len(A_rows) == 0:
        results = []
    else:
        A_rows = np.atleast_2d(A_rows)
        for row in A_rows:
            result, row_elapsed_s = one_row(row)
            total_elapsed_s += row_elapsed_s
            results.append(result)

    result_arr = np.array(results) if results else np.zeros((0, N))

    tm_current, tm_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return result_arr, (tm_peak / 1024), total_elapsed_s

def run_systolic_streaming_2_stages(A, B, arr_M, arr_N, arr_K, dview):


    M, K = len(A), len(A[0])
    N = len(B[0])

    n_row_tiles = (M + arr_M - 1) // arr_M
    n_col_tiles = (N + arr_N - 1) // arr_N
    n_k_tiles   = (K + arr_K - 1) // arr_K

   
    tiles = []
    
    for m in range(n_row_tiles):
        row_lo, row_hi = m * arr_M, min((m + 1) * arr_M, M)
        A_row_tile_full = A[row_lo:row_hi]
        M_tile = row_hi - row_lo
        for c in range(n_col_tiles):
            col_lo, col_hi = c * arr_N, min((c + 1) * arr_N, N)
            N_tile = col_hi - col_lo
            for k in range(n_k_tiles):
                k_lo, k_hi = k * arr_K, min((k + 1) * arr_K, K)
                K_tile = k_hi - k_lo
                A_tile = [row[k_lo:k_hi] for row in A_row_tile_full]
                B_tile = [row[col_lo:col_hi] for row in B[k_lo:k_hi]]
                tiles.append({
                    "m": m, "c": c,
                    "A_tile": A_tile, "B_tile": B_tile,
                    "M_tile": M_tile, "N_tile": N_tile, "K_tile": K_tile,
                })

    n_tiles = len(tiles)
    buffers = ["A_rows_ping", "A_rows_pong"]

    def dispatch(i):
        t = tiles[i]
        buf = buffers[i % 2]
        dview.scatter(buf, t["A_tile"], flatten=True)
        ar = dview.map_async(
            row_worker, dview[buf],
            [t["B_tile"]] * len(dview.targets),
            [t["K_tile"]] * len(dview.targets),
            [t["N_tile"]] * len(dview.targets)
        )
        return ar

    total_algorithm_s = 0.0
    total_mem_kb       = 0.0
    tile_count         = 0
    C_blocks           = {}   

    t0 = time.perf_counter()

    ar_current = dispatch(0)  

    for i in range(n_tiles):
        
        ar_next = dispatch(i + 1) if i + 1 < n_tiles else None   

        all_results = ar_current.get()                        
        t = tiles[i]

        results     = [r[0] for r in all_results]
        mem_kb      = [r[1] for r in all_results]
        tile_algo_s = max(r[2] for r in all_results)

        total_algorithm_s += tile_algo_s
        total_mem_kb      += sum(mem_kb)
        tile_count        += 1

        key = (t["m"], t["c"])
        if key not in C_blocks:
            C_blocks[key] = np.zeros((t["M_tile"], t["N_tile"]))
        C_blocks[key] += np.vstack(results)

        ar_current = ar_next  

    total_operation_s     = time.perf_counter() - t0
    total_communication_s = total_operation_s - total_algorithm_s

    C_row_blocks = []
    for m in range(n_row_tiles):
        C_row_blocks.append(np.hstack([C_blocks[(m, c)] for c in range(n_col_tiles)]))
    C = np.vstack(C_row_blocks)

    return C, total_algorithm_s, total_communication_s, total_operation_s, total_mem_kb, tile_count, n_row_tiles, n_col_tiles, n_k_tiles

## Streaming Algorithm (1 Node | 4 CPU Cores)

In [9]:
matrix_sizes = [128, 64, 32, 16, 8, 4]
arr_M, arr_K, arr_N = 4, 4, 4

np.random.seed(0)
cpu_1n4e = [] 

for matrix_size in matrix_sizes:

    M, K, N = [matrix_size] * 3
    
    A = np.random.rand(M, K).tolist()
    B = np.random.rand(K, N).tolist()
    C_ref = np.asarray(A) @ np.asarray(B)
   
    cluster = hfc.HyperFPGACluster(nodes=test_nodes[1], engines_per_node=4)

    print("CLEARING...")
    rc = None
    if rc is not None:
        rc.shutdown()
        await cluster.stop_cluster()
    await cluster.release_cluster()
    print("Node released.")
    print(f"\n\n==== STARTING A {M}x{N} MATRIX ====\n")

    try:
        await cluster.configure()
        cluster.create_profile(mpi=False)
        cluster.engine_timeout = 180
        
        rc = await cluster.start_and_connect()
        print(f" ---- Engines connected: {rc.ids} ---- ")

        dview = rc[:]
        dview.block = True

        time.sleep(20)
        C, algorithm_s, communication_s, total_s, mem_kb, tile_count, n_row_tiles, n_col_tiles, n_k_tiles = run_systolic_streaming_2_stages(
            A, B,
            arr_M, arr_N, arr_K,
            dview
        )
        time.sleep(20)


        print(f"Total tiles:         {tile_count}")
        print(f"Algorithm time:      {algorithm_s:.6f} s")
        print(f"Communication time:  {communication_s:.6f} s")
        print(f"Total operation:     {total_s:.6f} s")
        print(f"Total memory:        {mem_kb:.3f} KB")
        print(f"Max absolute error:  {np.max(np.abs(C - C_ref))}")

        cpu_1n4e.append({                         
            "matrix": matrix_size, "tiles": tile_count,
            "algorithm_s": algorithm_s, "communication_s": communication_s,
            "total_s": total_s, "mem_kb": mem_kb,
            "error": float(np.max(np.abs(C - C_ref))),
        })


    except Exception as e:
        print(f"[FAILED]: {e}")
        
    finally:
        if rc is not None:
            rc.shutdown()
            await cluster.stop_cluster()
        await cluster.release_cluster()
        print("Node released.")

ALL_RESULTS["CPU · 1 node x 4 cores"] = cpu_1n4e
print(f"\nAll runs complete.")

CLEARING...
Node released.


==== STARTING A 128x128 MATRIX ====

Starting 4 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787051806-2aap-client.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787051806-2aap-client.json
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787051806-2aap-engine.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787051806-2aap-engine.json
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=

 25%|##5       | 1/4 [00:00<?, ?engine/s]

 ---- Engines connected: [0, 1, 2, 3] ---- 
Total tiles:         32768
Algorithm time:      53.657686 s
Communication time:  2119.404756 s
Total operation:     2173.062441 s
Total memory:        163990.997 KB
Max absolute error:  4.973799150320701e-14
Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 14367, 'identifier': 'ipcontroller-1787051806-2aap-14149'}
Stopping engine(s): 1787051810


bash: line 1: kill: (4000) - No such process
bash: line 1: kill: (4031) - No such process
bash: line 1: kill: (4073) - No such process
bash: line 1: kill: (4115) - No such process


Node released.
CLEARING...
Node released.


==== STARTING A 64x64 MATRIX ====

Starting 4 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@192.168.0.6:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787054060-ff5k-client.json to mlabadm@192.168.0.6:.ipython/profile_ssh/security/ipcontroller-1787054060-ff5k-client.json
ensuring remote mlabadm@192.168.0.6:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787054060-ff5k-engine.json to mlabadm@192.168.0.6:.ipython/profile_ssh/security/ipcontroller-1787054060-ff5k-engine.json
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
Running `/opt/hfpga-ml311/bin/ipengine -

...

Node released.
CLEARING...
Node released.


==== STARTING A 4x4 MATRIX ====

Starting 4 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787054703-e3rb-client.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787054703-e3rb-client.json
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787054703-e3rb-engine.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787054703-e3rb-engine.json
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
Running `/opt/hfpga-ml311/bin/ipengine --p

 25%|##5       | 1/4 [00:00<?, ?engine/s]

 ---- Engines connected: [0, 1, 2, 3] ---- 
Total tiles:         1
Algorithm time:      0.001215 s
Communication time:  0.534196 s
Total operation:     0.535411 s
Total memory:        5.344 KB
Max absolute error:  0.0
Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 21964, 'identifier': 'ipcontroller-1787054703-e3rb-14149'}
Stopping engine(s): 1787054707


bash: line 1: kill: (6420) - No such process
bash: line 1: kill: (6463) - No such process
bash: line 1: kill: (6505) - No such process


Node released.

All runs complete.


## Streaming Algorithm (4 Nodes | 1 CPU Core Each)

In [4]:
matrix_sizes = [128, 64, 32, 16, 8, 4]
arr_M, arr_K, arr_N = 4, 4, 4
cpu_4n1e = []

np.random.seed(0)

results = []

for matrix_size in matrix_sizes:

    M, K, N = [matrix_size] * 3
    
    A = np.random.rand(M, K).tolist()
    B = np.random.rand(K, N).tolist()
    C_ref = np.asarray(A) @ np.asarray(B)

    
    cluster = hfc.HyperFPGACluster(nodes=test_nodes[0:4])

    rc = None
    try:
        await cluster.configure()
        cluster.create_profile(mpi=False)
        cluster.engine_timeout = 180

        rc = await cluster.start_and_connect()
        print(f" ---- Engines connected: {rc.ids} ---- ")

        dview = rc[:]
        dview.block = True

        time.sleep(20)
        (C, algorithm_s, communication_s, total_s, mem_kb,
         tile_count, n_row_tiles, n_col_tiles, n_k_tiles) = run_systolic_streaming_2_stages(
            A, B, arr_M, arr_N, arr_K, dview
        )
        time.sleep(20)

        error = float(np.max(np.abs(C - C_ref)))
        print(f"Tiles: {tile_count}\nAlgo: {algorithm_s:.4f}s\nComm: {communication_s:.4f}s\nTotal: {total_s:.4f}s\nMem: {mem_kb:.3f} KB")

        cpu_4n1e.append({
            "matrix": matrix_size,              
            "arr_M": arr_M, "arr_K": arr_K, "arr_N": arr_N,
            "tile_count": tile_count,
            "algorithm_s": algorithm_s, "communication_s": communication_s,
            "total_s": total_s, "mem_kb": mem_kb, "error": error,
        })

        results.append({
            "arr_M": arr_M, "arr_K": arr_K, "arr_N": arr_N,
            "tile_count": tile_count,
            "algorithm_s": algorithm_s, "communication_s": communication_s,
            "total_s": total_s, "mem_kb": mem_kb, "error": error, 
        })

    except Exception as e:
        print(f"[FAILED] matrix={matrix_size}: {e}")
    finally:
        if rc is not None:
            rc.shutdown()
            await cluster.stop_cluster()
        await cluster.release_cluster()
        print("Node released.")

ALL_RESULTS["CPU · 4 nodes x 1 core"] = cpu_4n1e
print(f"\nAll runs complete.")

Starting 4 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787056308-wg2e-client.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787056308-wg2e-client.json
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787056308-wg2e-engine.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787056308-wg2e-engine.json
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
ensuring remote mlabadm@192.168.0.6:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787056308-wg2e-client.json to mlabadm@192.168.0.6:.ipython/profile_ssh/security/ipcontroller-1787056308-wg2e-client.json
ensuring remote mlabadm@192.168.

 25%|##5       | 1/4 [00:00<?, ?engine/s]

 ---- Engines connected: [0, 1, 2, 3] ---- 
Tiles: 32768
Algo: 36.8412s
Comm: 1658.5797s
Total: 1695.4209s
Mem: 162816.562 KB
Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 26196, 'identifier': 'ipcontroller-1787056308-wg2e-26098'}
Stopping engine(s): 1787056310


bash: line 1: kill: (7078) - No such process
bash: line 1: kill: (7995) - No such process
bash: line 1: kill: (7668) - No such process
bash: line 1: kill: (7619) - No such process


Node released.
Starting 4 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787058084-2622-client.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787058084-2622-client.json
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787058084-2622-engine.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787058084-2622-engine.json
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
ensuring remote mlabadm@192.168.0.6:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787058084-2622-client.json to mlabadm@192.168.0.6:.ipython/profile_ssh/security/ipcontroller-1787058084-2622-client.json
ensuring remote m

...

Node released.
Starting 4 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787058640-wefu-client.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787058640-wefu-client.json
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787058640-wefu-engine.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787058640-wefu-engine.json
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
ensuring remote mlabadm@192.168.0.6:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787058640-wefu-client.json to mlabadm@192.168.0.6:.ipython/profile_ssh/security/ipcontroller-1787058640-wefu-client.json
ensuring remote m

 25%|##5       | 1/4 [00:00<?, ?engine/s]

 ---- Engines connected: [0, 1, 2, 3] ---- 
Tiles: 1
Algo: 0.0011s
Comm: 0.3100s
Total: 0.3111s
Mem: 5.344 KB
Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 32539, 'identifier': 'ipcontroller-1787058640-wefu-26098'}
Stopping engine(s): 1787058641


bash: line 1: kill: (8671) - No such process
bash: line 1: kill: (8631) - No such process


Node released.

All runs complete.


# GPU Part

### Sync `matmul_mali400.py` to each node

In [5]:
import os

def _install_matmul_mali400(source: str):
    """Runs on the remote engine: writes matmul_mali400.py locally and prepends it to sys.path."""
    import os as _os
    import sys as _sys
    import socket as _socket

    target_dir = _os.path.join(_os.path.expanduser("~"), ".cache", "matmul_mali400_sync")
    _os.makedirs(target_dir, exist_ok=True)
    target_path = _os.path.join(target_dir, "matmul_mali400.py")
    with open(target_path, "w", encoding="utf-8") as fh:
        fh.write(source)

    if target_dir in _sys.path:
        _sys.path.remove(target_dir)
    _sys.path.insert(0, target_dir)

    _sys.modules.pop("matmul_mali400", None)  # force re-import if a stale copy was already cached

    return _socket.gethostname(), target_path

## GPU Functions

In [6]:
def gpu_tile_batch_worker_bundled(bundle_list):
    import numpy as np
    import time
    import tracemalloc
    import matmul_mali400 as gpu

    tracemalloc.start()

    results = []
    total_gpu_ms = 0.0
    total_elapsed = 0.0

    for A_rows, B, K, N in bundle_list:
        if len(A_rows) == 0:
            results.append(np.zeros((0, N)))
            continue
        A_arr = np.atleast_2d(np.asarray(A_rows, dtype=np.float32))
        B_arr = np.asarray(B, dtype=np.float32)
        t0 = time.perf_counter()
        C_tile, gpu_ms = gpu.run_tile_gemm(A_arr, B_arr)
        total_elapsed += time.perf_counter() - t0
        total_gpu_ms += gpu_ms
        results.append(C_tile)

    tm_current, tm_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return results, total_gpu_ms, total_elapsed, (tm_peak / 1024)


def run_systolic_gpu_batched_2_stages(A, B, arr_M, arr_N, arr_K, dview, batch_size=4):
    import time
    M, K = len(A), len(A[0])
    N = len(B[0])

    n_row_tiles = (M + arr_M - 1) // arr_M
    n_col_tiles = (N + arr_N - 1) // arr_N
    n_k_tiles   = (K + arr_K - 1) // arr_K

    tiles = []
    for m in range(n_row_tiles):
        row_lo, row_hi = m * arr_M, min((m + 1) * arr_M, M)
        A_row_tile_full = A[row_lo:row_hi]
        M_tile = row_hi - row_lo
        for c in range(n_col_tiles):
            col_lo, col_hi = c * arr_N, min((c + 1) * arr_N, N)
            N_tile = col_hi - col_lo
            for k in range(n_k_tiles):
                k_lo, k_hi = k * arr_K, min((k + 1) * arr_K, K)
                K_tile = k_hi - k_lo
                A_tile = [row[k_lo:k_hi] for row in A_row_tile_full]
                B_tile = [row[col_lo:col_hi] for row in B[k_lo:k_hi]]
                tiles.append({
                    "m": m, "c": c,
                    "A_tile": A_tile, "B_tile": B_tile,
                    "M_tile": M_tile, "N_tile": N_tile, "K_tile": K_tile,
                })

    batches = [tiles[i:i + batch_size] for i in range(0, len(tiles), batch_size)]
    n_batches = len(batches)
    buffers = ["A_rows_ping", "A_rows_pong"]

    def dispatch(bi):
        if bi >= n_batches:
            return None
        batch = batches[bi]
        buf = buffers[bi % 2]
        bundled = [(t["A_tile"], t["B_tile"], t["K_tile"], t["N_tile"]) for t in batch]
        dview.scatter(buf, bundled, flatten=False)
        ar = dview.map_async(gpu_tile_batch_worker_bundled, dview[buf])
        return ar, batch

    total_algorithm_s = 0.0
    total_gpu_ms       = 0.0
    max_mem_kb          = 0.0   # CHANGED: track the max, not a running sum
    tile_count         = 0
    C_blocks           = {}

    t0 = time.perf_counter()
    ar_current, batch_current = dispatch(0)

    for bi in range(n_batches):
        nxt = dispatch(bi + 1)
        ar_next, batch_next = nxt if nxt is not None else (None, None)

        all_results = ar_current.get()

        batch_algo_s = max(r[2] for r in all_results)
        total_algorithm_s += batch_algo_s
        total_gpu_ms += sum(r[1] for r in all_results)
        max_mem_kb = max(max_mem_kb, max(r[3] for r in all_results))  # CHANGED

        idx = 0
        for results_list, _gpu_ms, _elapsed, _peak_kb in all_results:
            for r in results_list:
                t = batch_current[idx]
                idx += 1
                tile_count += 1
                key = (t["m"], t["c"])
                if key not in C_blocks:
                    C_blocks[key] = np.zeros((t["M_tile"], t["N_tile"]))
                C_blocks[key] += r

        ar_current, batch_current = ar_next, batch_next

    total_operation_s     = time.perf_counter() - t0
    total_communication_s = total_operation_s - total_algorithm_s

    C_row_blocks = []
    for m in range(n_row_tiles):
        C_row_blocks.append(np.hstack([C_blocks[(m, c)] for c in range(n_col_tiles)]))
    C = np.vstack(C_row_blocks)

    return (C, total_algorithm_s, total_communication_s, total_operation_s,
            total_gpu_ms, max_mem_kb, tile_count, n_row_tiles, n_col_tiles, n_k_tiles)

## Batched Streaming Algorithm (1 Node | 1 GPU)

In [7]:
matrix_sizes = [128, 64, 32, 16, 8, 4]
arr_M, arr_K, arr_N = 4, 4, 4
BATCH_SIZE = 8
gpu_1n = []


np.random.seed(0)


# STARTING THE EXPERIMENT
for matrix_size in matrix_sizes:
    
    M, K, N = [matrix_size] * 3
    
    A = np.random.rand(M, K).tolist()
    B = np.random.rand(K, N).tolist()
    C_ref = np.asarray(A) @ np.asarray(B)

    cluster = hfc.HyperFPGACluster(nodes=test_nodes[0])

    print(f"\n\n==== STARTING A {M}x{N} MATRIX ====\n")
    
    rc = None
    try:
        print(f" ---- Reserving {N_NODES} node(s) with 1 engine each... ---- ")
        await cluster.configure()
        cluster.create_profile()
        cluster.engine_timeout = 180

        print(" ---- Starting ipyparallel engines (one per board)... ---- ")
        rc = await cluster.start_and_connect()
        print(f" ---- Engines connected: {rc.ids} ---- ")

        dview = rc[:]
        dview.block = True

        print(" ---- Syncing matmul_mali400.py (local -> each node) ---- ")
        local_matmul_path = os.path.join(os.getcwd(), "matmul_mali400.py")
        with open(local_matmul_path, "r", encoding="utf-8") as f:
            matmul_mali400_src = f.read()

        sync_results = dview.apply_sync(_install_matmul_mali400, matmul_mali400_src)
        for host, path in sync_results:
            print(f"  - {host}: {path}")

        print(" ---- Initializing EGL once per engine (reused for every tile) ---- ")
        dview.execute("""
        import ctypes as _ctypes
        import socket as _socket
        import matmul_mali400 as gpu
        _egl_state = gpu.egl_init()
        _renderer = _ctypes.cast(gpu._gl("glGetString")(0x1F01), _ctypes.c_char_p).value.decode()
        _hostname = _socket.gethostname()
        """)
        for host, renderer in zip(dview.pull("_hostname"), dview.pull("_renderer")):
            print(f"  - {host}: {renderer}")

        time.sleep(20)
        (C, algorithm_s, communication_s, total_operation_s, total_gpu_ms, total_mem_kb,
         tile_count, n_row_tiles, n_col_tiles, n_k_tiles) = run_systolic_gpu_batched_2_stages(
            A, B, arr_M, arr_N, arr_K, dview, batch_size=BATCH_SIZE
        )
        time.sleep(20)

        error = float(np.max(np.abs(C - C_ref)))

        print(f"\nTiles: {tile_count}")
        print(f"Algorithm time: {algorithm_s:.6f} s")
        print(f"Communication time: {communication_s:.6f} s")
        print(f"Total operation time: {total_operation_s:.6f} s")
        print(f"Total GPU draw+readback time: {total_gpu_ms:.3f} ms")
        print(f"Total memory (peak, summed): {total_mem_kb:.3f} KB")
        print(f"Max absolute error: {error}")


        gpu_1n.append({
            "matrix": matrix_size,                
            "arr_M": arr_M, "arr_K": arr_K, "arr_N": arr_N,
            "tile_count": tile_count,
            "algorithm_s": algorithm_s, "communication_s": communication_s,
            "total_s": total_operation_s, "total_gpu_ms": total_gpu_ms,
            "total_mem_kb": total_mem_kb, "error": error,
        })


    except Exception as e:
        print(f"\n[FAILED] matrix:{matrix_size}: {e}")
    finally:
        print("\n ---- Releasing EGL context on each engine... ---- ")
        if rc is not None:
            try:
                dview.execute("gpu.egl_teardown(*_egl_state)")
            except Exception as teardown_err:
                print(f"  [WARNING] Could not cleanly release EGL: {teardown_err}")

        print("Releasing nodes...")
        if rc is not None:
            rc.shutdown()
            await cluster.stop_cluster()
        await cluster.release_cluster()
        print("Nodes released.")


ALL_RESULTS["GPU · 1 node x 1 GPU"] = gpu_1n
print(f"\nAll runs complete. {len(results)}/{len(matrix_sizes)} succeeded.")



==== STARTING A 128x128 MATRIX ====

 ---- Reserving 4 node(s) with 1 engine each... ---- 
 ---- Starting ipyparallel engines (one per board)... ---- 
Starting 1 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787059106-yosc-client.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787059106-yosc-client.json
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787059106-yosc-engine.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787059106-yosc-engine.json
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`


  0%|          | 0/1 [00:00<?, ?engine/s]

 ---- Engines connected: [0] ---- 
 ---- Syncing matmul_mali400.py (local -> each node) ---- 
  - hyperfpga-4ge21-0-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
 ---- Initializing EGL once per engine (reused for every tile) ---- 
  - hyperfpga-4ge21-0-1: Mali400

Tiles: 32768
Algorithm time: 181.445187 s
Communication time: 151.372272 s
Total operation time: 332.817460 s
Total GPU draw+readback time: 14611.556 ms
Total memory (peak, summed): 18.903 KB
Max absolute error: 0.614425121931184

 ---- Releasing EGL context on each engine... ---- 
Releasing nodes...
Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 33785, 'identifier': 'ipcontroller-1787059106-yosc-26098'}
Stopping engine(s): 1787059107
Nodes released.


==== STARTING A 64x64 MATRIX ====

 ---- Reserving 4 node(s) with 1 engine each... ---- 
 ---- Starting ipyparallel engines (one per board)... ---- 
Starting 1 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring rem

  0%|          | 0/1 [00:00<?, ?engine/s]

 ---- Engines connected: [0] ---- 
 ---- Syncing matmul_mali400.py (local -> each node) ---- 
  - hyperfpga-4ge21-0-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
 ---- Initializing EGL once per engine (reused for every tile) ---- 
  - hyperfpga-4ge21-0-1: Mali400

Tiles: 4096
Algorithm time: 22.861438 s
Communication time: 19.014770 s
Total operation time: 41.876208 s
Total GPU draw+readback time: 1901.141 ms
Total memory (peak, summed): 18.856 KB
Max absolute error: 0.3327366381118608

 ---- Releasing EGL context on each engine... ---- 
Releasing nodes...
Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 34826, 'identifier': 'ipcontroller-1787059506-c2ko-26098'}
Stopping engine(s): 1787059507
Nodes released.


==== STARTING A 32x32 MATRIX ====

 ---- Reserving 4 node(s) with 1 engine each... ---- 
 ---- Starting ipyparallel engines (one per board)... ---- 
Starting 1 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote 

...

 ---- Engines connected: [0] ---- 
 ---- Syncing matmul_mali400.py (local -> each node) ---- 
  - hyperfpga-4ge21-0-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
 ---- Initializing EGL once per engine (reused for every tile) ---- 
  - hyperfpga-4ge21-0-1: Mali400

Tiles: 64
Algorithm time: 0.388704 s
Communication time: 0.301627 s
Total operation time: 0.690331 s
Total GPU draw+readback time: 28.648 ms
Total memory (peak, summed): 18.810 KB
Max absolute error: 0.08667536608348758

 ---- Releasing EGL context on each engine... ---- 
Releasing nodes...
Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 35417, 'identifier': 'ipcontroller-1787059679-uls6-26098'}
Stopping engine(s): 1787059680
Nodes released.


==== STARTING A 8x8 MATRIX ====

 ---- Reserving 4 node(s) with 1 engine each... ---- 
 ---- Starting ipyparallel engines (one per board)... ---- 
Starting 1 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@

  0%|          | 0/1 [00:00<?, ?engine/s]

 ---- Engines connected: [0] ---- 
 ---- Syncing matmul_mali400.py (local -> each node) ---- 
  - hyperfpga-4ge21-0-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
 ---- Initializing EGL once per engine (reused for every tile) ---- 
  - hyperfpga-4ge21-0-1: Mali400

Tiles: 8
Algorithm time: 0.083460 s
Communication time: 0.038710 s
Total operation time: 0.122170 s
Total GPU draw+readback time: 5.314 ms
Total memory (peak, summed): 18.202 KB
Max absolute error: 0.04377490046896604

 ---- Releasing EGL context on each engine... ---- 
Releasing nodes...
Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 35652, 'identifier': 'ipcontroller-1787059743-bnsb-26098'}
Stopping engine(s): 1787059744
Nodes released.


==== STARTING A 4x4 MATRIX ====

 ---- Reserving 4 node(s) with 1 engine each... ---- 
 ---- Starting ipyparallel engines (one per board)... ---- 
Starting 1 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@19

  0%|          | 0/1 [00:00<?, ?engine/s]

 ---- Engines connected: [0] ---- 
 ---- Syncing matmul_mali400.py (local -> each node) ---- 
  - hyperfpga-4ge21-0-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
 ---- Initializing EGL once per engine (reused for every tile) ---- 
  - hyperfpga-4ge21-0-1: Mali400

Tiles: 1
Algorithm time: 0.046231 s
Communication time: 0.037459 s
Total operation time: 0.083691 s
Total GPU draw+readback time: 2.177 ms
Total memory (peak, summed): 6.583 KB
Max absolute error: 0.017093920551585873

 ---- Releasing EGL context on each engine... ---- 
Releasing nodes...
Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 35884, 'identifier': 'ipcontroller-1787059806-hqt4-26098'}
Stopping engine(s): 1787059807
Nodes released.

All runs complete. 6/6 succeeded.


## batched Streaming Algorithm (4 Nodes | 1 GPU Each)

In [8]:
matrix_sizes = [128, 64, 32, 16, 8, 4]
arr_M, arr_K, arr_N = 4, 4, 4
BATCH_SIZE = 32
gpu_4n = []

np.random.seed(0)


# STARTING THE EXPERIMENT
for matrix_size in matrix_sizes:
    
    M, K, N = [matrix_size] * 3
    
    A = np.random.rand(M, K).tolist()
    B = np.random.rand(K, N).tolist()
    C_ref = np.asarray(A) @ np.asarray(B)

    cluster = hfc.HyperFPGACluster(nodes=test_nodes[0:4])

    print(f"\n\n==== STARTING A {M}x{N} MATRIX ====\n")
    
    rc = None
    try:
        print(f" ---- Reserving {N_NODES} node(s) with 1 engine each... ---- ")
        await cluster.configure()
        cluster.create_profile()
        cluster.engine_timeout = 180

        print(" ---- Starting ipyparallel engines (one per board)... ---- ")
        rc = await cluster.start_and_connect()
        print(f" ---- Engines connected: {rc.ids} ---- ")

        dview = rc[:]
        dview.block = True

        print(" ---- Syncing matmul_mali400.py (local -> each node) ---- ")
        local_matmul_path = os.path.join(os.getcwd(), "matmul_mali400.py")
        with open(local_matmul_path, "r", encoding="utf-8") as f:
            matmul_mali400_src = f.read()

        sync_results = dview.apply_sync(_install_matmul_mali400, matmul_mali400_src)
        for host, path in sync_results:
            print(f"  - {host}: {path}")

        print(" ---- Initializing EGL once per engine (reused for every tile) ---- ")
        dview.execute("""
        import ctypes as _ctypes
        import socket as _socket
        import matmul_mali400 as gpu
        _egl_state = gpu.egl_init()
        _renderer = _ctypes.cast(gpu._gl("glGetString")(0x1F01), _ctypes.c_char_p).value.decode()
        _hostname = _socket.gethostname()
        """)
        for host, renderer in zip(dview.pull("_hostname"), dview.pull("_renderer")):
            print(f"  - {host}: {renderer}")

        time.sleep(20)
        (C, algorithm_s, communication_s, total_operation_s, total_gpu_ms, total_mem_kb,
         tile_count, n_row_tiles, n_col_tiles, n_k_tiles) = run_systolic_gpu_batched_2_stages(
            A, B, arr_M, arr_N, arr_K, dview, batch_size=BATCH_SIZE
        )
        time.sleep(20)

        error = float(np.max(np.abs(C - C_ref)))

        print(f"\nTiles: {tile_count}")
        print(f"Algorithm time: {algorithm_s:.6f} s")
        print(f"Communication time: {communication_s:.6f} s")
        print(f"Total operation time: {total_operation_s:.6f} s")
        print(f"Total GPU draw+readback time: {total_gpu_ms:.3f} ms")
        print(f"Total memory (peak, summed): {total_mem_kb:.3f} KB")
        print(f"Max absolute error: {error}")

        gpu_4n.append({
            "matrix": matrix_size,                
            "arr_M": arr_M, "arr_K": arr_K, "arr_N": arr_N,
            "tile_count": tile_count,
            "algorithm_s": algorithm_s, "communication_s": communication_s,
            "total_s": total_operation_s, "total_gpu_ms": total_gpu_ms,
            "total_mem_kb": total_mem_kb, "error": error,
        })

    except Exception as e:
        print(f"\n[FAILED]: {e}")
    finally:
        print("\n ---- Releasing EGL context on each engine... ---- ")
        if rc is not None:
            try:
                dview.execute("gpu.egl_teardown(*_egl_state)")
            except Exception as teardown_err:
                print(f"  [WARNING] Could not cleanly release EGL: {teardown_err}")

        print("Releasing nodes...")
        if rc is not None:
            rc.shutdown()
            await cluster.stop_cluster()
        await cluster.release_cluster()
        print("Nodes released.")


ALL_RESULTS["GPU · 4 nodes x 1 GPU"] = gpu_4n
print(f"\nAll runs complete. {len(results)}/{len(matrix_sizes)} succeeded.")



==== STARTING A 128x128 MATRIX ====

 ---- Reserving 4 node(s) with 1 engine each... ---- 
 ---- Starting ipyparallel engines (one per board)... ---- 
Starting 4 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787059870-0lhr-client.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787059870-0lhr-client.json
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787059870-0lhr-engine.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787059870-0lhr-engine.json
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
ensuring remote mlabadm@192.168.0.6:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1

 50%|#####     | 2/4 [00:00<?, ?engine/s]

 ---- Engines connected: [0, 1, 2, 3] ---- 
 ---- Syncing matmul_mali400.py (local -> each node) ---- 
  - hyperfpga-4ge21-0-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
  - hyperfpga-4ge21-1-0: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
  - hyperfpga-4ge21-1-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
  - hyperfpga-4ge21-1-3: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
 ---- Initializing EGL once per engine (reused for every tile) ---- 
  - hyperfpga-4ge21-0-1: Mali400
  - hyperfpga-4ge21-1-0: Mali400
  - hyperfpga-4ge21-1-1: Mali400
  - hyperfpga-4ge21-1-3: Mali400

Tiles: 32768
Algorithm time: 46.519003 s
Communication time: 55.128431 s
Total operation time: 101.647433 s
Total GPU draw+readback time: 14716.104 ms
Total memory (peak, summed): 18.952 KB
Max absolute error: 0.614425121931184

 ---- Releasing EGL context on each engine... ---- 
Releasing nodes...
Stopping controller
Controller stopped: {'exit_code': 0, 'p

 50%|#####     | 2/4 [00:00<?, ?engine/s]

 ---- Engines connected: [0, 1, 2, 3] ---- 
 ---- Syncing matmul_mali400.py (local -> each node) ---- 
  - hyperfpga-4ge21-0-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
  - hyperfpga-4ge21-1-0: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
  - hyperfpga-4ge21-1-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
  - hyperfpga-4ge21-1-3: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
 ---- Initializing EGL once per engine (reused for every tile) ---- 
  - hyperfpga-4ge21-0-1: Mali400
  - hyperfpga-4ge21-1-0: Mali400
  - hyperfpga-4ge21-1-1: Mali400
  - hyperfpga-4ge21-1-3: Mali400

Tiles: 4096
Algorithm time: 5.675339 s
Communication time: 6.820008 s
Total operation time: 12.495347 s
Total GPU draw+readback time: 1773.512 ms
Total memory (peak, summed): 18.763 KB
Max absolute error: 0.3327366381118608

 ---- Releasing EGL context on each engine... ---- 
Releasing nodes...
Stopping controller
Controller stopped: {'exit_code': 0, 'pid':

...

Nodes released.


==== STARTING A 4x4 MATRIX ====

 ---- Reserving 4 node(s) with 1 engine each... ---- 
 ---- Starting ipyparallel engines (one per board)... ---- 
Starting 4 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787060391-qj2r-client.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787060391-qj2r-client.json
ensuring remote mlabadm@192.168.0.3:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ipcontroller-1787060391-qj2r-engine.json to mlabadm@192.168.0.3:.ipython/profile_ssh/security/ipcontroller-1787060391-qj2r-engine.json
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`
ensuring remote mlabadm@192.168.0.6:.ipython/profile_ssh/security/ exists
sending /home/falmohammed/.ipython/profile_ssh/security/ip

 25%|##5       | 1/4 [00:00<?, ?engine/s]

 ---- Engines connected: [0, 1, 2, 3] ---- 
 ---- Syncing matmul_mali400.py (local -> each node) ---- 
  - hyperfpga-4ge21-0-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
  - hyperfpga-4ge21-1-0: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
  - hyperfpga-4ge21-1-1: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
  - hyperfpga-4ge21-1-3: /home/mlabadm/.cache/matmul_mali400_sync/matmul_mali400.py
 ---- Initializing EGL once per engine (reused for every tile) ---- 
  - hyperfpga-4ge21-0-1: Mali400
  - hyperfpga-4ge21-1-0: Mali400
  - hyperfpga-4ge21-1-1: Mali400
  - hyperfpga-4ge21-1-3: Mali400

Tiles: 1
Algorithm time: 0.046192 s
Communication time: 0.063455 s
Total operation time: 0.109647 s
Total GPU draw+readback time: 2.201 ms
Total memory (peak, summed): 6.536 KB
Max absolute error: 0.017093920551585873

 ---- Releasing EGL context on each engine... ---- 
Releasing nodes...
Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 38046

bash: line 1: kill: (9481) - No such process
bash: line 1: kill: (9446) - No such process


Nodes released.

All runs complete. 6/6 succeeded.


In [10]:
print(ALL_RESULTS)

{'CPU · 4 nodes x 1 core': [{'matrix': 128, 'arr_M': 4, 'arr_K': 4, 'arr_N': 4, 'tile_count': 32768, 'algorithm_s': 36.84122759710772, 'communication_s': 1658.57966295989, 'total_s': 1695.4208905569976, 'mem_kb': 162816.5625, 'error': 4.973799150320701e-14}, {'matrix': 64, 'arr_M': 4, 'arr_K': 4, 'arr_N': 4, 'tile_count': 4096, 'algorithm_s': 4.604377502333591, 'communication_s': 204.92219916464273, 'total_s': 209.52657666697633, 'mem_kb': 20352.515625, 'error': 1.4210854715202004e-14}, {'matrix': 32, 'arr_M': 4, 'arr_K': 4, 'arr_N': 4, 'tile_count': 512, 'algorithm_s': 0.5783593509586353, 'communication_s': 25.059894924004766, 'total_s': 25.6382542749634, 'mem_kb': 2544.65625, 'error': 5.329070518200751e-15}, {'matrix': 16, 'arr_M': 4, 'arr_K': 4, 'arr_N': 4, 'tile_count': 64, 'algorithm_s': 0.07167177599694696, 'communication_s': 3.509403022017068, 'total_s': 3.581074798014015, 'mem_kb': 318.609375, 'error': 1.7763568394002505e-15}, {'matrix': 8, 'arr_M': 4, 'arr_K': 4, 'arr_N': 4, '